In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import os
import re
import numpy as np

#### Scarapping

In [ ]:
BASE_URL = "https://aqarmap.com.eg/en/for-sale/egypt/"
OUTPUT_FILE = "aqarmap_full22.csv"
COLUMNS = [
    "Listing_ID", "Type", "Address", "Price", "Price_Per_M2", "Area",
    "Rooms", "Bathrooms", "Floor", "Year_Built", "Finish",
    "View", "Payment_Method", "Seller_Type",
    "Amenities", "Description", "Link"
]

def start_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(f"--user-data-dir=C:/temp/chrome_profile_{int(time.time())}")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(options=options)

def extract_full_address(link):
    try:
        part = link.split("for-sale-")[-1].strip("/")
        address = part.replace("-", " ")
        address = re.sub(r'https?://\S+', '', address).strip()
        if address.startswith("projects"):
            address = address.replace("projects", "", 1).strip()
        return address
    except:
        return "N/A"

def scrape_listing(driver, link):
    row = {col: "N/A" for col in COLUMNS}
    row["Link"] = link
    row["Address"] = extract_full_address(link)

    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
        lines = [l.strip() for l in body_text.split("\n") if l.strip()]
    except:
        return row

    def detect_type(lines, body_text):
        month_pattern = re.compile(
            r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\b',
            re.IGNORECASE
        )
        for i, l in enumerate(lines):
            if month_pattern.search(l) and i > 0:
                candidate = lines[i - 1].lower()
                types = [
                    "apartment", "flat", "villa", "duplex", "penthouse", 
                    "studio", "chalet", "office", "clinic", "medical", 
                    "shop", "commercial", "land", "plot", "roof", "building"
                ]
                for t in types:
                    if t in candidate: 
                        return t.title()
                break
                
        body_lower = body_text.lower()
        if "roof" in body_lower: return "Roof"
        if "penthouse" in body_lower: return "Penthouse"
        if "duplex" in body_lower: return "Duplex"
        if "chalet" in body_lower: return "Chalet"
        if "villa" in body_lower: return "Villa"
        if "apartment" in body_lower or "flat" in body_lower: return "Apartment"
        return "Unknown"

    row["Type"] = detect_type(lines, body_text)

    for l in lines:
        if "EGP" in l and "/m" not in l and re.search(r'\d', l):
            row["Price"] = l
            break
            
    for l in lines:
        if "/m²" in l or "EGP/M²" in l:
            row["Price_Per_M2"] = l
            break
            
    for l in lines:
        if "m²" in l and "EGP" not in l:
            row["Area"] = l
            break
            
    for l in lines:
        if re.search(r'\d+\s*(room|bed)', l, re.IGNORECASE):
            row["Rooms"] = l
            break
            
    for l in lines:
        if "bathroom" in l.lower() or "bath" in l.lower():
            row["Bathrooms"] = l
            break

    body_lower = body_text.lower()

    if row["Area"] == "N/A":
        area_match = re.search(r'(?:size\s*\(in\s*meters\)|area)\s*[:\n\-]?\s*(\d+)', body_lower)
        if area_match:
            row["Area"] = f"{area_match.group(1)} m²"
        else:
            alt_area = re.search(r'(\d+)\s*(m²|m2|sqm|metre)', body_lower)
            if alt_area: 
                row["Area"] = f"{alt_area.group(1)} m²"

    if row["Rooms"] == "N/A":
        rooms_match = re.search(r'(?:bedrooms?|rooms?)\s*[:\n\-]?\s*(\d+)', body_lower)
        if rooms_match:
            row["Rooms"] = f"{rooms_match.group(1)}"
        else:
            alt_rooms = re.search(r'(\d+)\s*(bedrooms?|rooms?|beds?)', body_lower)
            if alt_rooms: 
                row["Rooms"] = f"{alt_rooms.group(1)}"

    if row["Bathrooms"] == "N/A":
        baths_match = re.search(r'(?:bathrooms?|baths?)\s*[:\n\-]?\s*(\d+)', body_lower)
        if baths_match:
            row["Bathrooms"] = f"{baths_match.group(1)}"
        else:
            alt_baths = re.search(r'(\d+)\s*(bathrooms?|baths?)', body_lower)
            if alt_baths: 
                row["Bathrooms"] = f"{alt_baths.group(1)}"
                
    if row["Price"] == "N/A":
        price_match = re.search(r'([\d,]+)\s*(EGP)', body_text, re.IGNORECASE)
        if price_match: 
            row["Price"] = f"{price_match.group(1)} EGP"

    if row["Rooms"] == "N/A" or row["Bathrooms"] == "N/A":
        for i, l in enumerate(lines):
            if ("m²" in l or "m2" in l.lower() or "sqm" in l.lower()) and re.search(r'\d', l):
                if i + 1 < len(lines):
                    next_line = lines[i+1].strip()
                    if next_line.isdigit() and int(next_line) < 20:
                        if row["Rooms"] == "N/A": 
                            row["Rooms"] = next_line
                if i + 2 < len(lines):
                    second_next = lines[i+2].strip()
                    if second_next.isdigit() and int(second_next) < 20:
                        if row["Bathrooms"] == "N/A": 
                            row["Bathrooms"] = second_next
                break

    key_map = {
        "floor": "Floor", 
        "year built": "Year_Built", 
        "seller type": "Seller_Type",
        "listing id": "Listing_ID", 
        "view": "View", 
        "payment method": "Payment_Method",
        "finish": "Finish"
    }
    
    for i, l in enumerate(lines):
        l_lower = l.lower()
        for key, col in key_map.items():
            if l_lower == key and i + 1 < len(lines):
                if row[col] == "N/A": 
                    row[col] = lines[i + 1]

    amenity_keywords = ["elevator", "water meter", "natural gas", "security", "swimming pool", "parking", "garden", "balcony"]
    found_amenities = [a.title() for a in amenity_keywords if a in body_text.lower()]
    row["Amenities"] = ", ".join(found_amenities) if found_amenities else "N/A"

    try:
        desc_el = driver.find_element(By.XPATH, "//div[contains(@class,'listing-description')] | //section[@id='description']")
        row["Description"] = desc_el.text.strip()[:500]
    except:
        pass

    return row

seen_links = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_old = pd.read_csv(OUTPUT_FILE)
        if not df_old.empty and "Link" in df_old.columns:
            seen_links = set(df_old["Link"].dropna().unique())
            print(f"Recovery: {len(seen_links)} links loaded.")
    except Exception as e:
        print(f"Error loading file: {e}")
else:
    pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

driver = start_driver()
wait = WebDriverWait(driver, 15)

try:
    for page in range(350, 1001):
        print(f"\n================ PAGE {page} ================")
        
        try:
            driver.get(BASE_URL + f"?page={page}")
            wait.until(EC.presence_of_element_located((By.CLASS_NAME, "listing-card")))
            time.sleep(2)
        except Exception:
            print("Page error. Restarting driver...")
            try: 
                driver.quit()
            except: 
                pass
            driver = start_driver()
            wait = WebDriverWait(driver, 15)
            continue

        elements = driver.find_elements(By.XPATH, "//a[contains(@href,'/listing/')]")
        page_links = []
        for el in elements:
            try:
                href = el.get_attribute("href")
                if href and href not in page_links:
                    page_links.append(href)
            except: 
                continue

        for i, link in enumerate(page_links):
            if link in seen_links:
                continue 

            print(f"   [{i+1}/{len(page_links)}] Processing: {link}")
            
            try:
                driver.get(link)
                wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                time.sleep(1.5)
                
                row = scrape_listing(driver, link)
                
                pd.DataFrame([row]).to_csv(
                    OUTPUT_FILE, mode="a", header=False, index=False, encoding="utf-8-sig"
                )
                seen_links.add(link)
                
                print(f"    Saved: {row['Type']} | Area: {row['Area']} | Rooms: {row['Rooms']} | Baths: {row['Bathrooms']}")
                
            except Exception as e:
                print(f"    Error: {e}")
                if "session" in str(e).lower() or "no such window" in str(e).lower():
                    try: 
                        driver.quit()
                    except: 
                        pass
                    driver = start_driver()
                    wait = WebDriverWait(driver, 15)
                continue
                
finally:
    try: 
        driver.quit()
    except: 
        pass
    print("SCRAPING FINISHED")

In [27]:
data = pd.read_csv("aqarmap.csv")
print(data.head())


   Listing_ID       Type                                        Address  \
0  EG-6901132      Villa                    north coast resorts riviera   
1  EG-6671452  Apartment          cairo 6th of october compounds murooj   
2  EG-6903778       Land             cairo 6th of october ltws t lshrqy   
3  EG-6897924     Chalet  north coast resorts solare resort misr italia   
4  EG-6898957  Apartment                 cairo new heliopolis lhy lthny   

           Price   Price_Per_M2     Area    Rooms   Bathrooms   Floor  \
0  6,800,000 EGP  19,429 EGP/M²   350 m²  4 rooms  3 bathroom     NaN   
1  5,000,000 EGP  37,594 EGP/M²   133 m²  3 rooms  2 bathroom       2   
2  5,500,000 EGP   5,314 EGP/M²  1035 m²      NaN         NaN     NaN   
3  9,974,100 EGP  75,561 EGP/M²   132 m²  2 rooms  2 bathroom  Ground   
4  1,600,000 EGP  13,333 EGP/M²   120 m²  2 rooms  3 bathroom       3   

   Year_Built                              Finish         View Payment_Method  \
0      2009.0                

In [28]:
len(data)

5374

#### Preprocessing Stage (Preparing for Data Integration)

In [29]:
# Handle inconsistent layouts by removing rows where all key features are missing
# and filtering out unidentified property types.
data.replace("N/A", np.nan, inplace=True)
cols_to_check = ['Area','Rooms', 'Bathrooms']
data = data[data['Type'] != 'Unknown']
data_cleaned = data.dropna(subset=cols_to_check, how='all')

# to integrate with the other code
columns_to_drop = ["Description", "Listing_ID", "Year_Built", "Seller_Type", "Link"]
data_cleaned = data_cleaned.drop(columns=columns_to_drop, errors='ignore')


#Extract the number of amenities as a numerical feature
data_cleaned['Amenities_Count'] = data_cleaned['Amenities'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) and x != 'N/A' else 0
)

# Label Encoding: Map unfinished/semi-finished to 0 and all finished types to 1
data_cleaned['Finish'] = data_cleaned['Finish'].apply(
    lambda x: 0 if x in ['Semi Finished', 'Without Finish'] else 1
)

#Clean Payment_Method and encode as binary (1 for installment availability, 0 for cash only)
data_cleaned['Is_Installments'] = data_cleaned['Payment_Method'].apply(
    lambda x: 1 if 'Installments' in str(x) else 0
)
data_cleaned.drop(columns=['Payment_Method'], inplace=True, errors='ignore')

# Cleaning numerical columns from text suffixes (EGP, m², rooms, etc.)
def clean_numeric(value):
    if pd.isna(value) or value == 'N/A':
        return np.nan
    value = str(value).replace(',', '')  
    match = re.search(r'\d+\.?\d*', value)  
    return float(match.group()) if match else np.nan

numeric_cols = ['Price', 'Price_Per_M2', 'Area', 'Rooms', 'Bathrooms']
for col in numeric_cols:
    data_cleaned[col] = data_cleaned[col].apply(clean_numeric)

# Convert Floor column (handle 'Ground' as 0 and others as numbers)
data_cleaned['Floor'] = data_cleaned['Floor'].replace('Ground', '0')
data_cleaned['Floor'] = pd.to_numeric(data_cleaned['Floor'].str.extract('(\d+)', expand=False), errors='coerce')

#save dataaa
data_cleaned.to_csv('aqarmap_final.csv', index=False, encoding='utf-8-sig')

#Read dataa
data=pd.read_csv("aqarmap_final.csv")
len(data)

<>:43: SyntaxWarning: invalid escape sequence '\d'
<>:43: SyntaxWarning: invalid escape sequence '\d'
C:\TEMP\ipykernel_22368\690176679.py:43: SyntaxWarning: invalid escape sequence '\d'
  data_cleaned['Floor'] = pd.to_numeric(data_cleaned['Floor'].str.extract('(\d+)', expand=False), errors='coerce')


3532

In [39]:
len(data)

3532

In [6]:
print(data.columns)
print(data.Finish.value_counts())
data.Type.value_counts()

Index(['Type', 'Address', 'Price', 'Price_Per_M2', 'Area', 'Rooms',
       'Bathrooms', 'Floor', 'Finish', 'View', 'Amenities', 'Amenities_Count',
       'Is_Installments'],
      dtype='object')
Finish
1    3483
0      49
Name: count, dtype: int64


Type
Apartment     2215
Duplex         361
Villa          293
Chalet         145
Penthouse      105
Roof            97
Office          76
Studio          67
Commercial      60
Land            58
Medical         28
Building        21
Shop             6
Name: count, dtype: int64

In [37]:
len(data.Address.unique())

1327

In [36]:
data.View.value_counts()

View
Main Street    1466
Garden          798
Side Street     366
Other           321
Pool            179
Corner          141
Sea View         63
Plaza            46
Back             45
Lake             36
Seaview          19
Nile View        11
Club              8
Golf Course       6
Name: count, dtype: int64

In [32]:
data.head(20)

,Type,Address,Price,Price_Per_M2,Area,Rooms,Bathrooms,Floor,Finish,View,Amenities,Amenities_Count,Is_Installments
0,Villa,north coast resorts riviera,6800000.0,19429.0,350.0,4.0,3.0,NaN,1,Garden,Garden,1,1
1,Apartment,cairo 6th of october compounds murooj,5000000.0,37594.0,133.0,3.0,2.0,2.0,1,Main Street,NaN,0,0
2,Land,cairo 6th of october ltws t lshrqy,5500000.0,5314.0,1035.0,NaN,NaN,NaN,1,Main Street,NaN,0,0
3,Chalet,north coast resorts solare resort misr italia,9974100.0,75561.0,132.0,2.0,2.0,0.0,1,Lake,"Security, Garden, Solar",3,1
4,Apartment,cairo new heliopolis lhy lthny,1600000.0,13333.0,120.0,2.0,3.0,3.0,1,Main Street,NaN,0,1
5,Land,cairo 6th of october green belt,160000000.0,4762.0,33600.0,NaN,NaN,NaN,1,Main Street,NaN,0,0
6,Apartment,cairo nasr city abbas el akkad,6500000.0,32500.0,200.0,3.0,3.0,11.0,1,Main Street,"Elevator, Water Meter, Security, Balcony",4,0
7,Studio,cairo new cairo compounds sarai croons,3185000.0,46159.0,69.0,1.0,1.0,2.0,1,Main Street,NaN,0,0
8,Chalet,north coast resorts virginia plaza resort solid,1578000.0,28691.0,55.0,2.0,1.0,7.0,1,Pool,NaN,0,1
9,Apartment,cairo new cairo hy skn ljm lmryky,7350000.0,31013.0,237.0,4.0,3.0,2.0,1,Garden,Garden,1,0
